In [92]:
from helper.ingest import load_faq_data, build_index
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
from helper.evaluation_utils import llm_structured, calc_price,llm_structured_retry, map_progress,RAGWithUsage,calc_total_price
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json
from toyaikit.llm import OpenAIClient

In [2]:
load_dotenv()
openai_client = OpenAI()

In [3]:
documents = load_faq_data()

In [4]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
user_prompt = json.dumps(doc)


In [11]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['I just found this course — is it still okay to sign up and start now?', 'Can latecomers still join the class, or is it too late already?', 'If I’m starting the course now, can I still work toward getting the certificate?', 'Do I have to submit the project before the submission window closes to get certified?', 'What’s the latest point I can join and still be eligible for the certificate?']


In [15]:
print(result.questions)

['I just found this course — is it still okay to sign up and start now?', 'Can latecomers still join the class, or is it too late already?', 'If I’m starting the course now, can I still work toward getting the certificate?', 'Do I have to submit the project before the submission window closes to get certified?', 'What’s the latest point I can join and still be eligible for the certificate?']


In [16]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)


['Can I still join the course if I found it late?', 'Is it too late to start the course now?', 'If I join after the course started, can I still get a certificate?', 'Do I have to submit the project before submissions close to get the certificate?', 'What do I need to do to qualify for a certificate if I join late?']


In [17]:
usage.input_tokens, usage.output_tokens

(207, 84)

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00037799999999999997,
 'total_cost': 0.00053325}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records


[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I join late?',
  'document': '74eb249bbf'}]

In [20]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [21]:


ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [22]:
usages

[ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=88, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=295),
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=357),
 ResponseUsage(input_tokens=315, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=125, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=440),
 ResponseUsage(input_tokens=379, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=489),
 ResponseUsage(input_tokens=381, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=102, output_tokens

In [23]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [24]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [25]:
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08679750000000001

In [26]:
df_ground_truth = pd.DataFrame(ground_truth)

In [27]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [28]:
df_ground_truth

,question,document
0,Is it too late to start the course if I just f...,74eb249bbf
1,Can I still join the course after it has alrea...,74eb249bbf
2,"If I enroll late, do I still have a chance to ...",74eb249bbf
3,Do I need to submit the project before submiss...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf
...,...,...
560,Why does pip keep giving me requests 2.28 when...,4b30b918bc
561,Is there a way to install requests straight fr...,4b30b918bc
562,My setup is stuck on requests 2.28 and lancedb...,4b30b918bc
563,Could the 401 Client Error mean my key access ...,4b30b918bc


In [29]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [30]:
documents_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [31]:
index = build_index(documents_llm)

In [32]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [33]:
q = ground_truth[0]
q


{'question': 'Is it too late to start the course if I just found it now?',
 'document': '74eb249bbf'}

In [34]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [35]:
results

[{'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](ht

In [36]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

0fab61eca2 == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [37]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 0, 1, 0, 0]

In [38]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [39]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Where should I look for the livestream link before an Office Hours session starts?


[1, 0, 0, 0, 0]

In [40]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [41]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [42]:
relevance_total_text

[[0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [43]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [44]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [45]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [46]:
relevance_total 

[[0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

In [47]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [48]:
hit_rate(relevance_total)

0.879646017699115

In [49]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [50]:
mrr(relevance_total)

0.7383775811209434

In [51]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [52]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.879646017699115, 'mrr': 0.7383775811209434}

In [53]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [54]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8938053097345132, 'mrr': 0.802300884955752}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9097345132743363, 'mrr': 0.793362831858407}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.879646017699115, 'mrr': 0.7383775811209434}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8530973451327434, 'mrr': 0.7094100294985247}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.815929203539823, 'mrr': 0.6755457227138636}


In [55]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [56]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [57]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.976991,0.882891
3,1.0,2.0,0.1,0.976991,0.882891
19,2.0,4.0,0.2,0.976991,0.882891
18,2.0,4.0,0.1,0.976991,0.881062
33,5.0,10.0,0.1,0.975221,0.880737
34,5.0,10.0,0.2,0.976991,0.880177
4,1.0,2.0,0.2,0.973451,0.875988
20,2.0,4.0,0.5,0.971681,0.872861
6,1.0,4.0,0.1,0.968142,0.865192
7,1.0,4.0,0.2,0.966372,0.864749


In [58]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [59]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [60]:
assistant

In [61]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [62]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [66]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [67]:
assistant.total_cost()

0.6159420000000001

In [68]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [69]:
df_answers

,question,answer_llm,answer_orig,document
0,Is it too late to start the course if I just f...,"Yes, you can still join. You can start wheneve...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still join the course after it has alrea...,"Yes, you can still join the course after it ha...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I enroll late, do I still have a chance to ...",Yes — but only if you submit your project whil...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit the project before submiss...,"Yes. To get certified, you need to submit your...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What’s the deadline for project submission if ...,There isn’t a fixed project deadline given her...,"Yes, but if you want to receive a certificate,...",74eb249bbf
...,...,...,...,...
560,Why does pip keep giving me requests 2.28 when...,Because `pip` is likely resolving to an older ...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
561,Is there a way to install requests straight fr...,Yes — install it directly from the GitHub rele...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
562,My setup is stuck on requests 2.28 and lancedb...,Use this exact command:\n\n```bash\npip instal...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
563,Could the 401 Client Error mean my key access ...,Yes — a 401 Client Error usually means an auth...,"If you encounter a 401 Client Error, it may in...",4b30b918bc


In [70]:
answers

[{'question': 'Is it too late to start the course if I just found it now?',
  'answer_llm': 'Yes, you can still join. You can start whenever you want, but if you want a certificate, you need to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still join the course after it has already started?',
  'answer_llm': 'Yes, you can still join the course after it has already started, but to receive a certificate you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I enroll late, do I still have a chance to get the certificate?',
  'answer_llm': 'Yes — but only if you submit your project while

In [72]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [73]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [74]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [76]:
rec = answers[0]


In [77]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)


In [78]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it is not too late to start, and certificate eligibility depends on submitting the project while submissions are still accepted/open. This is semantically equivalent.', score='good')

In [79]:
calc_price(usage)

{'input_cost': 0.00022425, 'output_cost': 0.0002565, 'total_cost': 0.00048075}

In [80]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [81]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [82]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [83]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [84]:
df_eval = pd.DataFrame(evaluations)

In [87]:
calc_total_price(usages)

0.4416832499999999

In [88]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 544/565 = 96.28%


In [89]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
14,Should I put questions in the live chat during...,489dd1c9d9,bad,The AI answer captures the key instruction tha...
33,Is the Capstone project the only thing that re...,9f689c185f,bad,The AI answer correctly states that passing th...
42,"Is there an upcoming session of the course, an...",bd31146b0e,bad,The ground truth says the upcoming session is ...
44,When can I expect this course to run again?,bd31146b0e,bad,The ground truth gives a specific expected run...
47,How do I know which video goes with a specific...,31456f4b5f,bad,The AI answer does not convey the guidance fro...


In [90]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

### Agent Evaluation

In [93]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

In [95]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [96]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [97]:
instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [98]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [99]:
 result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it too late to start the course if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"too late to start the course just found it now start late enrollment late join"}', call_id='call_HobMbjN97ShlMnJ4v5YFqPrr', name='search', type='function_call', id='fc_024d39e12b9b5c5c006a50f7b5bd6481a3b667cd487b2c0424', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_HobMbjN97ShlMnJ4v5YFqPrr',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/do

In [100]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [101]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"too late to start the course just found it now start late enrollment late join"}'}]

In [103]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [104]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it too late to start the course if I just found it now?',
 'answer_agent': 'No — it’s not too late to start.\n\nThe course materials and videos are available, and you can start whenever you want. If you want a certificate, the key thing is to submit your capstone project while submissions are still being accepted.\n\nA couple of notes:\n- You can also work through the homework while the submission form is open.\n- Homework deadlines are strict: no late homework submissions after the form closes.\n\nIf you want, I can also point you to the recommended way to start the course.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"too late to start the course just found it now start late enrollment late join"}'}],
 'cost': Decimal('0.0013935'),
 'document': '74eb249bbf'}

In [105]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [106]:
with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [107]:
df_agent = pd.DataFrame(agent_answers)

In [108]:
df_agent["cost"].sum()

Decimal('0.06415950')

In [109]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [110]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [111]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [112]:
def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [113]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer is broadly aligned with the ground truth: it says it is not too late to start the course and mentions that materials are available. However, it adds details about homework deadlines and starting steps that are not in the original answer, and it does not explicitly capture the key condition from the FAQ that to receive a certificate, the project must be submitted while submissions are still being accepted. Still, the core idea that you can start now is present, so the answer is mostly correct.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included the key idea of starting late / too late to start. However, there was only one search, which is reasonable, but the final answer ended up missing the certificate-specific condition from the ground truth. The tool call was useful but not fully aligned with the exact FAQ answer.', trajectory_score='good')

In [114]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [115]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [116]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [117]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [118]:
calc_total_price(usages)

0.053431499999999986

In [119]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    50
Name: count, dtype: int64

In [120]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [121]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)